In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2-T1 interleaved-public layout

Reloads one persisted shared terminal, writes messages 0/1, and runs two VAE decodes plus sixteen receiver encodes; it does not generate a terminal or run OFF.


In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL='https://github.com/RICHAAARC/SC-SSTW.git'; SOURCE_BRANCH='c2a-2a-colab-preparation'; SOURCE=Path('/content/c2t1_interleaved_source')
if SOURCE.exists():
    if subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()!=REPOSITORY_URL: raise RuntimeError('unexpected source origin')
else:
    subprocess.run(['git','init',str(SOURCE)],check=True); subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',REPOSITORY_URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_BRANCH],check=True); subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','--force','FETCH_HEAD'],check=True)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True); subprocess.run(['ffmpeg','-version'],check=True)


In [ ]:
from datetime import datetime, timezone
CONFIG=SOURCE/'runtime/c2t1/c2t1_interleaved_run.json'; RUN_ID='c2t1_interleaved_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); OUTPUT=Path('/content/drive/MyDrive/Video-WM/C2T1_Interleaved')/RUN_ID
if OUTPUT.exists(): raise FileExistsError(OUTPUT)


In [ ]:
import os, signal
command=[sys.executable,'-u','-m','runtime.c2t1.run_interleaved','--config',str(CONFIG),'--output',str(OUTPUT)]; LOG=OUTPUT.parent/f'{RUN_ID}.launcher.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
with LOG.open('w',encoding='utf-8') as log:
    process=subprocess.Popen(command,cwd=SOURCE,start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout: print(line,end=''); log.write(line); log.flush()
        returncode=process.wait()
    except BaseException:
        try: process.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid,signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
        raise
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file'); print('Drive log:',LOG)
if returncode: raise subprocess.CalledProcessError(returncode,command)
